# 读写文件

到目前为止，我们讨论了如何处理数据，
以及如何构建、训练和测试深度学习模型。
然而，有时我们希望保存训练的模型，
以备将来在各种环境中使用（比如在部署中进行预测）。
此外，当运行一个耗时较长的训练过程时，
最佳的做法是定期保存中间结果，
以确保在服务器电源被不小心断掉时，我们不会损失几天的计算结果。
因此，现在是时候学习如何加载和存储权重向量和整个模型了。

## (**加载和保存张量**)

对于单个张量，我们可以直接调用`load`和`save`函数分别读写它们。
这两个函数都要求我们提供一个名称，`save`要求将要保存的变量作为输入。


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

我们现在可以将存储在文件中的数据读回内存。


In [4]:
x2 = torch.load('x-file')
x2

/tmp/ipykernel_131179/1444019752.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x2 = torch.load('x-file')


tensor([0, 1, 2, 3])

In [21]:
#出现警告
#torch.load('mydict')  ##默认等价于：  ##torch.load('mydict', weights_only=False)
#它会使用 Python pickle 反序列化。
#恶意构造的文件可能在加载时执行任意代码，因此不要加载来源不可信的 .pt、.pth 或其他存档文件。
# PyTorch 从 2.6 开始已默认采用更受限制的 weights_only=True

我们可以[**存储一个张量列表，然后把它们读回内存。**]


In [7]:
y = torch.zeros(4)
torch.save([x, y],'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

/tmp/ipykernel_131179/2924237495.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x2, y2 = torch.load('x-files')


(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我们甚至可以(**写入或读取从字符串映射到张量的字典**)。
当我们要读取或写入模型中的所有权重时，这很方便。


In [8]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

/tmp/ipykernel_131179/2269971588.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mydict2 = torch.load('mydict')


{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## [**加载和保存模型参数**]

保存单个权重向量（或其他张量）确实有用，
但是如果我们想保存整个模型，并在以后加载它们，
单独保存每个向量则会变得很麻烦。
毕竟，我们可能有数百个参数散布在各处。
因此，深度学习框架提供了内置函数来保存和加载整个网络。
【**需要注意的一个重要细节是，这将保存模型的参数而不是保存整个模型**】。
例如，如果我们有一个3层多层感知机，我们需要单独指定架构。
因为模型本身可以包含任意代码，所以模型本身难以序列化。
因此，为了恢复模型，我们需要用代码生成架构，
然后从磁盘加载参数。
让我们从熟悉的多层感知机开始尝试一下。


In [10]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

接下来，我们[**将模型的参数存储在一个叫做“mlp.params”的文件中。**]


In [16]:
net.state_dict() 

OrderedDict([('hidden.weight',
              tensor([[-0.0348,  0.1671,  0.0944,  ..., -0.1391,  0.2198,  0.0039],
                      [ 0.0440,  0.1116, -0.0788,  ...,  0.0366,  0.2112, -0.0767],
                      [ 0.1078, -0.1877,  0.1703,  ..., -0.2084, -0.0434, -0.1198],
                      ...,
                      [ 0.0514,  0.0698, -0.0719,  ...,  0.2135, -0.2110,  0.0268],
                      [-0.2067, -0.0536,  0.0545,  ..., -0.1196,  0.0154, -0.1562],
                      [-0.0338, -0.0816,  0.0634,  ..., -0.1036, -0.0269,  0.0199]])),
             ('hidden.bias',
              tensor([-0.0491, -0.1358, -0.0681,  0.2110, -0.2055, -0.0597,  0.2011,  0.0614,
                       0.1218,  0.1798,  0.0551,  0.1685, -0.0230, -0.1043,  0.0917, -0.1139,
                      -0.0988,  0.2136, -0.1691, -0.1603,  0.0400, -0.2055, -0.0615, -0.1247,
                      -0.1394,  0.2220,  0.1431, -0.1743, -0.1208,  0.1706, -0.1094,  0.2075,
                       0.0809,

In [18]:
torch.save(net.state_dict(), 'mlp.params')

为了恢复模型，我们[**实例化了原始多层感知机模型的一个备份。**]
这里我们不需要随机初始化模型参数，而是(**直接读取文件中存储的参数。**)


In [24]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()   
##将模型切换到评估模式，也就是告诉 PyTorch：接下来主要用于验证或预测，不是在训练。
#它在内部会设置：clone.training = False
#并递归地将所有子模块也切换为评估模式。

/tmp/ipykernel_131179/4217989229.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clone.load_state_dict(torch.load('mlp.params'))


MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [27]:
#为什么 Jupyter 会显示模型结构？
#val() 会返回模型自身：
#result = clone.eval()
#result is clone
# True
#所以当它是代码 Cell 最后一行时，Jupyter 会显示：
#MLP(
#  (hidden): Linear(...)
#  (output): Linear(...)
#)

#这不是额外创建了一个模型，只是 clone.eval() 返回了已经切换到评估模式的 clone。

In [26]:
#主要是训练和预测行为不同的层会受影响
#Dropout
#训练模式下，Dropout 会随机丢弃部分神经元：
#model.train()
#评估模式下则停止随机丢弃：
#model.eval()

#BatchNorm
#训练模式下，BatchNorm 使用当前批次的均值和方差，并更新运行统计量。
#评估模式下，它使用训练期间积累的运行均值和方差，不再更新统计量。

#.eval() 不会关闭梯度
#需要注意：clone.eval()
#只切换层的运行模式，不会：
#- 关闭自动求导
#- 冻结模型参数
#- 阻止反向传播

#进行预测时通常配合：
#clone.eval()

#with torch.no_grad():
#    Y_clone = clone(X)

#或者使用更适合纯推理的：
#clone.eval()

#with torch.inference_mode():
#    Y_clone = clone(X)

由于两个实例具有相同的模型参数，在输入相同的`X`时，
两个实例的计算结果应该相同。
让我们来验证一下。


In [29]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## 小结

* `save`和`load`函数可用于张量对象的文件读写。
* 我们可以通过参数字典保存和加载网络的全部参数。
* 保存架构必须在代码中完成，而不是在参数中完成。

## 练习

1. 即使不需要将经过训练的模型部署到不同的设备上，存储模型参数还有什么实际的好处？
1. 假设我们只想复用网络的一部分，以将其合并到不同的网络架构中。比如想在一个新的网络中使用之前网络的前两层，该怎么做？
1. 如何同时保存网络架构和参数？需要对架构加上什么限制？


[Discussions](https://discuss.d2l.ai/t/1839)
